# Module 06: Autograd - The Gradient Engine

Welcome to Module 06! Today you'll awaken the gradient engine and unlock automatic differentiation.


In [1]:
#| default_exp foundation.autograd
#| export


import numpy as np
rng = np.random.default_rng(7)
from typing import Optional, List, Tuple
import sys
import os

sys.path.append(r"D:\DS\Data Science\TinyTorch\projects\my-tinytorch-systems")
from tinytorch.foundation.tensor import Tensor

# Constants for numerical differentiation
EPSILON = 1e-7  # Small perturbation for numerical gradient computation

##  Implementation: Building the Autograd Engine

Let's implement the autograd system step by step. We'll enhance the existing Tensor class and create supporting infrastructure.


### Function Base Class - The Foundation of Autograd

The Function class is the foundation that makes autograd possible. Every differentiable operation (addition, multiplication, etc.) inherits from this class.


In [2]:
#| export
class Function:
    """
    Base class for differentiable operations.

    Every operation that needs gradients (add, multiply, matmul, etc.)
    will inherit from this class and implement the apply() method.

    **Key Concepts:**
    - **saved_tensors**: Store inputs needed for backward pass
    - **apply()**: Compute gradients using chain rule

    **Example Usage:**
    ```python
    class AddBackward(Function):
        def apply(self, grad_output):
            # Addition distributes gradients equally
            return grad_output, grad_output
    ```
    """

    def __init__(self, *tensors):
        """
        Initialize function with input tensors.

        Args:
            *tensors: Input tensors that will be saved for backward pass
        """
        self.saved_tensors = tensors

    def apply(self, grad_output):
        """
        Compute gradients for inputs.

        Args:
            grad_output: Gradient flowing backward from the output

        Returns:
            Tuple of gradients for each input tensor

        **Must be implemented by subclasses**
        """
        raise NotImplementedError("Each Function must implement apply() method")

### Operation Functions - Implementing Gradient Rules

Now we'll implement specific operations that compute gradients correctly. Each operation has mathematical rules for how gradients flow backward.

In [4]:
#| export
def _reduce_broadcast_grad(grad, original_shape):
    """
    Reduce gradient to match original tensor shape after broadcasting.

    TODO: Implement gradient shape reduction for broadcast operations.

    APPROACH:
    1. Remove leading dimensions: while grad has more dims than original, sum axis=0
    2. Collapse singleton dimensions: where original had size 1, sum with keepdims=True
    """
    
    # Step 1: Remove leading dimensions that weren't in original tensor
    # Example: grad (32, 128) with original (128,) → sum over axis 0
    while grad.ndim > len(original_shape):
        grad = grad.sum(axis=0)
    
    # Step 2: Collapse dimensions where original had size 1
    # Example: grad (10, 5) with original (10, 1) → sum over axis 1 with keepdims
    for i in range(len(original_shape)):
        if original_shape[i] == 1 and grad.shape[i] > 1:
            grad = grad.sum(axis=i, keepdims=True)
    
    return grad
    

## Unit Test: Broadcast Gradient Reduction¶
This test validates our gradient reduction helper handles all broadcasting scenarios.

In [5]:
def test_unit_reduce_broadcast_grad():
    """🔬 Test _reduce_broadcast_grad helper function."""
    print("🔬 Unit Test: _reduce_broadcast_grad...")
    
    # Test 1: Remove leading dimension
    grad = np.ones((32, 128))
    original_shape = (128,)
    reduced = _reduce_broadcast_grad(grad, original_shape)
    assert reduced.shape == (128,), f"Expected (128,), got {reduced.shape}"
    assert np.allclose(reduced, np.ones(128) * 32), "Should sum over batch dimension"
    print("  ✓ Leading dimension reduction works")
    
    # Test 2: Collapse singleton dimension
    grad = np.ones((10, 5))
    original_shape = (10, 1)
    reduced = _reduce_broadcast_grad(grad, original_shape)
    assert reduced.shape == (10, 1), f"Expected (10, 1), got {reduced.shape}"
    assert np.allclose(reduced, np.ones((10, 1)) * 5), "Should sum over singleton axis"
    print("  ✓ Singleton dimension reduction works")
    
    # Test 3: Both operations
    grad = np.ones((32, 10, 5))
    original_shape = (10, 1)
    reduced = _reduce_broadcast_grad(grad, original_shape)
    assert reduced.shape == (10, 1), f"Expected (10, 1), got {reduced.shape}"
    expected = np.ones((10, 1)) * 32 * 5  # Sum over batch and last dim
    assert np.allclose(reduced, expected), "Should handle multiple reductions"
    print("  ✓ Multiple dimension reduction works")
    
    # Test 4: No broadcasting (should return unchanged)
    grad = np.ones((10, 5))
    original_shape = (10, 5)
    reduced = _reduce_broadcast_grad(grad, original_shape)
    assert reduced.shape == (10, 5), f"Expected (10, 5), got {reduced.shape}"
    assert np.allclose(reduced, grad), "Should return unchanged when shapes match"
    print("  ✓ No-op case works")
    
    # Test 5: Scalar case (reduce to scalar)
    grad = np.ones((32, 128))
    original_shape = ()
    reduced = _reduce_broadcast_grad(grad, original_shape)
    assert reduced.shape == (), f"Expected scalar, got {reduced.shape}"
    assert np.allclose(reduced, 32 * 128), "Should sum to scalar"
    print("  ✓ Scalar reduction works")
    
    print("✅ _reduce_broadcast_grad works correctly!")

if __name__ == "__main__":
    test_unit_reduce_broadcast_grad()

🔬 Unit Test: _reduce_broadcast_grad...
  ✓ Leading dimension reduction works
  ✓ Singleton dimension reduction works
  ✓ Multiple dimension reduction works
  ✓ No-op case works
  ✓ Scalar reduction works
✅ _reduce_broadcast_grad works correctly!


### AddBackward - Gradient Rules for Addition

Addition is the simplest gradient operation: gradients flow unchanged to both inputs.

In [6]:
#| export
class AddBackward(Function):
    """
    Gradient computation for tensor addition.
    """

    def apply(self, grad_output):
        """
        Compute gradients for addition.

        Args:
            grad_output: Gradient flowing backward from output

        Returns:
            Tuple of (grad_a, grad_b) for the two inputs

        **Mathematical Foundation:**
        - ∂(a+b)/∂a = 1 → grad_a = grad_output
        - ∂(a+b)/∂b = 1 → grad_b = grad_output
        """
        
        a, b = self.saved_tensors
        grad_a = grad_b = None

        # Gradient for first input
        if isinstance(a, Tensor) and a.requires_grad:
            grad_a = grad_output
            # Handle broadcasting: reduce gradient to match original shape
            grad_a = _reduce_broadcast_grad(grad_a, a.data.shape)

        # Gradient for second input
        if isinstance(b, Tensor) and b.requires_grad:
            grad_b = grad_output
            # Handle broadcasting: reduce gradient to match original shape
            grad_b = _reduce_broadcast_grad(grad_b, b.data.shape)

        return grad_a, grad_b
 

### MulBackward - Gradient Rules for Element-wise Multiplication

Element-wise multiplication follows the product rule of calculus.

In [8]:
#| export
class MulBackward(Function):
    """
    Gradient computation for tensor multiplication.
    """

    def apply(self, grad_output):
        """
        Compute gradients for multiplication.

        Args:
            grad_output: Gradient flowing backward from output

        Returns:
            Tuple of (grad_a, grad_b) for the two inputs

        **Mathematical Foundation:**
        - ∂(a*b)/∂a = b → grad_a = grad_output * b
        - ∂(a*b)/∂b = a → grad_b = grad_output * a
        """
        
        a, b = self.saved_tensors
        grad_a = grad_b = None

        # Gradient for first input: grad_output * b
        if isinstance(a, Tensor) and a.requires_grad:
            if isinstance(b, Tensor):
                grad_a = grad_output * b.data
            else:
                grad_a = grad_output * b
            # Handle broadcasting: reduce gradient to match original shape
            grad_a = _reduce_broadcast_grad(grad_a, a.data.shape)

        # Gradient for second input: grad_output * a
        if isinstance(b, Tensor) and b.requires_grad:
            grad_b = grad_output * a.data
            # Handle broadcasting: reduce gradient to match original shape
            grad_b = _reduce_broadcast_grad(grad_b, b.data.shape)

        return grad_a, grad_b
        

### SubBackward - Gradient Rules for Subtraction

Subtraction is mathematically simple but important for operations like normalization.

In [9]:
#| export
class SubBackward(Function):
    """
    Gradient computation for tensor subtraction.

    **Mathematical Rule:** If z = a - b, then ∂z/∂a = 1 and ∂z/∂b = -1
    """

    def apply(self, grad_output):
        """
        Compute gradients for subtraction.

        Returns:
            Tuple of (grad_a, grad_b) where grad_b is negated
        """
       
        a, b = self.saved_tensors
        grad_a = grad_b = None

        if isinstance(a, Tensor) and a.requires_grad:
            grad_a = grad_output  # ∂(a-b)/∂a = 1
            # Handle broadcasting: reduce gradient to match original shape
            grad_a = _reduce_broadcast_grad(grad_a, a.data.shape)

        if isinstance(b, Tensor) and b.requires_grad:
            grad_b = -grad_output  # ∂(a-b)/∂b = -1 (note the negative!)
            # Handle broadcasting: reduce gradient to match original shape
            grad_b = _reduce_broadcast_grad(grad_b, b.data.shape)

        return grad_a, grad_b
        

### DivBackward - Gradient Rules for Division

Division requires the quotient rule from calculus.


In [10]:
#| export
class DivBackward(Function):
    """
    Gradient computation for tensor division.

    **Mathematical Rule:** If z = a / b, then:
    - ∂z/∂a = 1/b
    - ∂z/∂b = -a/b²
    """

    def apply(self, grad_output):
        """
        Compute gradients for division using quotient rule.

        Returns:
            Tuple of (grad_a, grad_b)
        """
        
        a, b = self.saved_tensors
        grad_a = grad_b = None

        if isinstance(a, Tensor) and a.requires_grad:
            # ∂(a/b)/∂a = 1/b
            if isinstance(b, Tensor):
                grad_a = grad_output / b.data
            else:
                grad_a = grad_output / b
            # Handle broadcasting: reduce gradient to match original shape
            grad_a = _reduce_broadcast_grad(grad_a, a.data.shape)

        if isinstance(b, Tensor) and b.requires_grad:
            # ∂(a/b)/∂b = -a/b²
            grad_b = -grad_output * a.data / (b.data ** 2)
            # Handle broadcasting: reduce gradient to match original shape
            grad_b = _reduce_broadcast_grad(grad_b, b.data.shape)

        return grad_a, grad_b
        

### MatmulBackward - Gradient Rules for Matrix Multiplication

Matrix multiplication has more complex gradient rules based on matrix calculus.


In [11]:
#| export
class MatmulBackward(Function):
    """
    Gradient computation for matrix multiplication.

    **Mathematical Rule:** If Z = A @ B, then:
    - ∂Z/∂A = grad_Z @ B.T
    - ∂Z/∂B = A.T @ grad_Z
    """

    def apply(self, grad_output):
        """
        Compute gradients for matrix multiplication.

        Args:
            grad_output: Gradient flowing backward from output

        Returns:
            Tuple of (grad_a, grad_b) for the two matrix inputs
        """
        
        a, b = self.saved_tensors
        grad_a = grad_b = None

        # Gradient for first input: grad_output @ b.T
        if isinstance(a, Tensor) and a.requires_grad:
            if b.data.ndim >= 2:
                # Batched: transpose only the last two dims
                b_T = np.swapaxes(b.data, -2, -1)
                grad_a = np.matmul(grad_output, b_T)
            else:
                # 1D b: A(m,k) @ b(k,) -> out(m,)
                # grad_A = outer(grad_output, b): (m,) x (k,) -> (m, k)
                grad_a = np.outer(grad_output, b.data)

        # Gradient for second input: a.T @ grad_output
        if isinstance(b, Tensor) and b.requires_grad:
            if a.data.ndim >= 2:
                # Batched: transpose only the last two dims
                a_T = np.swapaxes(a.data, -2, -1)
                grad_b = np.matmul(a_T, grad_output)
            else:
                # 1D a: a(k,) @ B(k,n) -> out(n,)
                # grad_B = outer(a, grad_output): (k,) x (n,) -> (k, n)
                grad_b = np.outer(a.data, grad_output)

        return grad_a, grad_b
        

In [ ]:
#| export
class TransposeBackward(Function):
    """
    Gradient computation for transpose operation.

    **Mathematical Rule:** If Y = X.T, then:
    - ∂Y/∂X = grad_Y.T
    """

    def __init__(self, tensor, dim0, dim1):
        """
        Args:
            tensor: Input tensor
            dim0: First dimension to swap (None for default)
            dim1: Second dimension to swap (None for default)
        """
        super().__init__(tensor)
        self.dim0 = dim0
        self.dim1 = dim1

    def apply(self, grad_output):
        """
        Compute gradient for transpose.

        Args:
            grad_output: Gradient flowing backward from output

        Returns:
            Tuple with single gradient for input tensor
        """
        
        x, = self.saved_tensors
        grad_x = None

        if isinstance(x, Tensor) and x.requires_grad:
            # Transpose gradient using the same dims
            if self.dim0 is None and self.dim1 is None:
                # Default: transpose last two dimensions
                if grad_output.ndim < 2:
                    grad_x = grad_output.copy()
                else:
                    axes = list(range(grad_output.ndim))
                    axes[-2], axes[-1] = axes[-1], axes[-2]
                    grad_x = np.transpose(grad_output, axes)
            else:
                # Specific dimensions: swap them back
                axes = list(range(grad_output.ndim))
                axes[self.dim0], axes[self.dim1] = axes[self.dim1], axes[self.dim0]
                grad_x = np.transpose(grad_output, axes)

        return (grad_x,)
        

In [12]:
#| export
class PermuteBackward(Function):
    """
    Gradient computation for arbitrary axis permutation (general transpose).

    """

    def __init__(self, tensor, axes):
        """
        Args:
            tensor: Input tensor
            axes: Tuple of axis indices defining the permutation
        """
        super().__init__(tensor)
        self.axes = axes
        # Compute inverse permutation: if axes[i] = j, then inverse_axes[j] = i
        self.inverse_axes = tuple(np.argsort(axes))

    def apply(self, grad_output):
        """
        Compute gradient for permutation.

        The gradient is permuted back using the inverse permutation.
        
        """
        
        x, = self.saved_tensors
        grad_x = None

        if isinstance(x, Tensor) and x.requires_grad:
            # Permute gradient back to original axis order
            grad_x = np.transpose(grad_output, self.inverse_axes)

        return (grad_x,)
        ### END SOLUTION

In [14]:
#| export
class SliceBackward(Function):
    """
    Gradient computation for tensor slicing/indexing operations.
    """

    def __init__(self, tensor, key):
        """
        Args:
            tensor: Original tensor being sliced
            key: Slicing key (index, slice, tuple of slices, etc.)
        """
        super().__init__(tensor)
        self.key = key
        self.original_shape = tensor.shape

    def apply(self, grad_output):
        """
        Compute gradient for slicing operation.

        Args:
            grad_output: Gradient flowing backward from sliced output

        Returns:
            Tuple with single gradient for input tensor
        """
        
        tensor, = self.saved_tensors
        grad_input = None

        if isinstance(tensor, Tensor) and tensor.requires_grad:
            # Create gradient array with same shape as original tensor
            grad_input = np.zeros(self.original_shape, dtype=np.float32)

            # Place gradients back into the sliced positions
            # This is the inverse of the forward slicing operation
            grad_input[self.key] = grad_output

        return (grad_input,)
        

In [15]:
#| export
class ReshapeBackward(Function):
    """
    Gradient computation for reshape operation.
    """

    def __init__(self, tensor, original_shape):
        """
        Args:
            tensor: Input tensor
            original_shape: Shape before reshape
        """
        super().__init__(tensor)
        self.original_shape = original_shape

    def apply(self, grad_output):
        """
        Compute gradient for reshape.

        Args:
            grad_output: Gradient flowing backward from output

        Returns:
            Tuple with single gradient for input tensor
        """
        
        x, = self.saved_tensors
        grad_x = None

        if isinstance(x, Tensor) and x.requires_grad:
            # Reshape gradient back to original shape
            grad_x = grad_output.reshape(self.original_shape)

        return (grad_x,)
        

### SumBackward - Gradient Rules for Reduction Operations

Sum operations reduce tensor dimensions, so gradients must be broadcast back.

In [17]:
#| export
class SumBackward(Function):
    """
    Gradient computation for tensor sum.
    """

    def __init__(self, tensor, axis=None, keepdims=False):
        super().__init__(tensor)
        self.axis = axis
        self.keepdims = keepdims

    def apply(self, grad_output):
        """
        Compute gradients for sum operation.

        Args:
            grad_output: Gradient flowing backward from output

        Returns:
            Tuple containing gradient for the input tensor
        """
        
        tensor, = self.saved_tensors

        if isinstance(tensor, Tensor) and tensor.requires_grad:
            # For axis-reduced sums, expand grad_output back along the summed
            # axis before broadcasting, so each row/column gets its own gradient.
            if self.axis is not None and not self.keepdims:
                grad_output = np.expand_dims(grad_output, axis=self.axis)
            return np.ones_like(tensor.data) * grad_output,
        return None,
       

###  Unit Test: Function Classes

This test validates our Function classes compute gradients correctly.

In [18]:
def test_unit_function_classes():
    """ Test Function classes."""
    print(" Unit Test: Function Classes...")

    # Note: We manually set requires_grad here because enable_autograd() hasn't been
    # called yet. This tests the Backward classes in isolation before the full
    # autograd system is enabled.

    # Test AddBackward
    a = Tensor([1, 2, 3])
    a.requires_grad = True  # Set manually (enable_autograd not called yet)
    b = Tensor([4, 5, 6])
    b.requires_grad = True
    add_func = AddBackward(a, b)
    grad_output = np.array([1, 1, 1])
    grad_a, grad_b = add_func.apply(grad_output)
    assert np.allclose(grad_a, grad_output), f"AddBackward grad_a failed: {grad_a}"
    assert np.allclose(grad_b, grad_output), f"AddBackward grad_b failed: {grad_b}"

    # Test MulBackward
    mul_func = MulBackward(a, b)
    grad_a, grad_b = mul_func.apply(grad_output)
    assert np.allclose(grad_a, b.data), f"MulBackward grad_a failed: {grad_a}"
    assert np.allclose(grad_b, a.data), f"MulBackward grad_b failed: {grad_b}"

    # Test MatmulBackward
    a_mat = Tensor([[1, 2], [3, 4]])
    a_mat.requires_grad = True
    b_mat = Tensor([[5, 6], [7, 8]])
    b_mat.requires_grad = True
    matmul_func = MatmulBackward(a_mat, b_mat)
    grad_output = np.ones((2, 2))
    grad_a, grad_b = matmul_func.apply(grad_output)
    assert grad_a.shape == a_mat.shape, f"MatmulBackward grad_a shape: {grad_a.shape}"
    assert grad_b.shape == b_mat.shape, f"MatmulBackward grad_b shape: {grad_b.shape}"

    print("✅ Function classes work correctly!")

###  Unit Test: Broadcasting in Gradients
This test validates that gradient reduction works correctly when operations
involve broadcasting. This is crucial for real-world training where bias terms,
normalization parameters, and other operations broadcast over batches.

In [20]:
def test_unit_broadcast_gradients():
    """ Test gradient broadcasting reduction."""
    print(" Unit Test: Broadcasting in Gradients...")
    
    # Scenario 1: Bias-like broadcasting (most common case)
    # Shape: (batch, features) + (features,) → (batch, features)
    
    x = Tensor(rng.standard_normal((4, 3)))
    x.requires_grad=True
    bias = Tensor(np.ones(3))
    bias.requires_grad=True
    
    add_func = AddBackward(x, bias)
    grad_output = np.ones((4, 3))
    grad_x, grad_bias = add_func.apply(grad_output)

    # Check shapes
    assert grad_x.shape == x.data.shape, \
        f"Expected grad_x shape {x.data.shape}, got {grad_x.shape}"
    assert grad_bias.shape == bias.data.shape, \
        f"Expected grad_bias shape {bias.data.shape}, got {grad_bias.shape}"
    
    # Check values: bias gradient should sum over batch dimension
    expected_bias_grad = np.ones(3) * 4  # Sum of 4 ones = 4
    assert np.allclose(grad_bias, expected_bias_grad), \
        f"Expected bias grad {expected_bias_grad}, got {grad_bias}"
    
    print("  ✓ Bias-like broadcasting works")
    
    # Scenario 2: Scalar broadcasting
    # Shape: (3, 4) + scalar → (3, 4)
    x = Tensor(rng.standard_normal((3, 4)))
    x.requires_grad=True
    scalar_val = 5.0
    
    add_func = AddBackward(x, scalar_val)
    grad_output = np.ones((3, 4))
    grad_x, grad_scalar = add_func.apply(grad_output)
    
    assert grad_x.shape == x.data.shape, \
        f"Expected grad_x shape {x.data.shape}, got {grad_x.shape}"
    # Scalar gradients don't get reduced (not a Tensor)
    
    print("  ✓ Scalar broadcasting works")
    
    # Scenario 3: Multiple dimension broadcasting
    # Shape: (32, 10, 5) + (10, 1) → (32, 10, 5)
    x = Tensor(rng.standard_normal((32, 10, 5)))
    x.requires_grad=True
    y = Tensor(rng.standard_normal((10, 1)))
    y.requires_grad=True
    
    mul_func = MulBackward(x, y)
    grad_output = np.ones((32, 10, 5))
    grad_x, grad_y = mul_func.apply(grad_output)
    
    assert grad_x.shape == x.data.shape, \
        f"Expected grad_x shape {x.data.shape}, got {grad_x.shape}"
    assert grad_y.shape == y.data.shape, \
        f"Expected grad_y shape {y.data.shape}, got {grad_y.shape}"
    
    print("  ✓ Multi-dimension broadcasting works")
    
    # Scenario 4: Test all operations (Add, Mul, Sub, Div)
    a = Tensor(rng.standard_normal((8, 16)))
    a.requires_grad=True
    b = Tensor(rng.standard_normal(16))
    b.requires_grad=True
    
    # Test Addition
    add_func = AddBackward(a, b)
    grad_a, grad_b = add_func.apply(np.ones((8, 16)))
    assert grad_b.shape == (16,), f"AddBackward: Expected (16,), got {grad_b.shape}"
    
    # Test Multiplication
    mul_func = MulBackward(a, b)
    grad_a, grad_b = mul_func.apply(np.ones((8, 16)))
    assert grad_b.shape == (16,), f"MulBackward: Expected (16,), got {grad_b.shape}"
    
    # Test Subtraction
    sub_func = SubBackward(a, b)
    grad_a, grad_b = sub_func.apply(np.ones((8, 16)))
    assert grad_b.shape == (16,), f"SubBackward: Expected (16,), got {grad_b.shape}"
    
    # Test Division
    div_func = DivBackward(a, b)
    grad_a, grad_b = div_func.apply(np.ones((8, 16)))
    assert grad_b.shape == (16,), f"DivBackward: Expected (16,), got {grad_b.shape}"
    
    print("  ✓ All operations handle broadcasting correctly")
    
    # Scenario 5: Real-world case - Linear layer gradient
    # Simulates: output = input @ weight + bias
    # where bias is (out_features,) and output is (batch, out_features)
    batch_size, out_features = 32, 128
    output_grad = rng.standard_normal((batch_size, out_features))
    bias = Tensor(np.zeros(out_features))
    bias.requires_grad=True
    
    # In real Linear layer, bias gradient comes from output gradient
    x_w = Tensor(np.zeros((batch_size, out_features)))
    x_w.requires_grad=True
    add_func = AddBackward(
        x_w,
        bias
    )
    _, grad_bias = add_func.apply(output_grad)
    
    assert grad_bias.shape == (out_features,), \
        f"Linear layer bias: Expected ({out_features},), got {grad_bias.shape}"
    
    # Verify gradient is sum over batch
    expected = output_grad.sum(axis=0)
    assert np.allclose(grad_bias, expected), \
        "Bias gradient should equal sum over batch dimension"
    
    print("  ✓ Real-world Linear layer scenario works")
    
    print("✅ Broadcasting gradient tests pass!")

if __name__ == "__main__":
    test_unit_broadcast_gradients()

 Unit Test: Broadcasting in Gradients...
  ✓ Bias-like broadcasting works
  ✓ Scalar broadcasting works
  ✓ Multi-dimension broadcasting works
  ✓ All operations handle broadcasting correctly
  ✓ Real-world Linear layer scenario works
✅ Broadcasting gradient tests pass!


##  Enhancing Tensor with Autograd Capabilities

Now we'll enhance the existing Tensor class to use these gradient functions and build computation graphs automatically.


In [21]:
#| export
class ReLUBackward(Function):
    """
    Gradient computation for ReLU activation.

    ReLU: f(x) = max(0, x)
    Derivative: f'(x) = 1 if x > 0, else 0
    """

    def __init__(self, input_tensor):
        """Initialize with input tensor."""
        super().__init__(input_tensor)

    def apply(self, grad_output):
        """
        Compute gradient for ReLU.
        """
        
        tensor, = self.saved_tensors

        if isinstance(tensor, Tensor) and tensor.requires_grad:
            # ReLU gradient: 1 if x > 0, else 0
            relu_grad = (tensor.data > 0).astype(np.float32)
            return grad_output * relu_grad,
        return None,### END SOLUTION

In [22]:
#| export
class SigmoidBackward(Function):
    """
    Gradient computation for sigmoid activation.

    Sigmoid: σ(x) = 1/(1 + exp(-x))
    Derivative: σ'(x) = σ(x) * (1 - σ(x))
    """

    def __init__(self, input_tensor, output_tensor):
        """
        Initialize with both input and output.

        Args:
            input_tensor: Original input to sigmoid
            output_tensor: Output of sigmoid (saves recomputation)
        """
        super().__init__(input_tensor)
        self.output_data = output_tensor.data

    def apply(self, grad_output):
        """
        Compute gradient for sigmoid.
        """
       
        tensor, = self.saved_tensors

        if isinstance(tensor, Tensor) and tensor.requires_grad:
            # σ'(x) = σ(x) * (1 - σ(x))
            sigmoid_grad = self.output_data * (1 - self.output_data)
            return grad_output * sigmoid_grad,
        return None,
        

In [23]:
#| export
class TanhBackward(Function):
    """
    Gradient computation for tanh activation.

    Tanh: tanh(x) = (eˣ - e⁻ˣ) / (eˣ + e⁻ˣ)
    Derivative: tanh'(x) = 1 - tanh(x)²
    """

    def __init__(self, input_tensor, output_tensor):
        """
        Initialize with both input and output.

        Args:
            input_tensor: Original input to tanh
            output_tensor: Output of tanh (saves recomputation)
        """
        super().__init__(input_tensor)
        self.output_data = output_tensor.data

    def apply(self, grad_output):
        """
        Compute gradient for tanh.
        """
        
        tensor, = self.saved_tensors

        if isinstance(tensor, Tensor) and tensor.requires_grad:
            # tanh'(x) = 1 - tanh(x)²
            tanh_grad = 1 - self.output_data ** 2
            return grad_output * tanh_grad,
        return None,
        

In [24]:
#| export
class SoftmaxBackward(Function):
    """
    Gradient computation for softmax activation.

    Softmax: softmax(x)[i] = exp(x[i]) / sum(exp(x))
    Derivative: ∂softmax/∂x[i] = softmax[i] * (δ[i,j] - softmax[j])
    """

    def __init__(self, input_tensor, output_tensor, dim=-1):
        """
        Initialize with input, output, and dimension.

        Args:
            input_tensor: Original input to softmax
            output_tensor: Output of softmax (needed for gradient)
            dim: Dimension along which softmax was applied
        """
        super().__init__(input_tensor)
        self.output_data = output_tensor.data
        self.dim = dim

    def apply(self, grad_output):
        """
        Compute gradient for softmax.
        """
        
        tensor, = self.saved_tensors

        if isinstance(tensor, Tensor) and tensor.requires_grad:
            # Compute sum(grad_output * softmax) along the softmax dimension
            sum_term = np.sum(grad_output * self.output_data, axis=self.dim, keepdims=True)

            # Softmax gradient: softmax * (grad_output - sum_term)
            grad_x = self.output_data * (grad_output - sum_term)

            return (grad_x,)
        return (None,)
      

In [25]:
#| export
class GELUBackward(Function):
    """
    Gradient computation for GELU activation.

    GELU: f(x) = x * Φ(x) where Φ is the CDF of standard normal
    Approximation: gelu(x) ≈ 0.5 * x * (1 + tanh(√(2/π) * (x + 0.044715 * x³)))
    """

    def __init__(self, input_tensor):
        """Initialize with input tensor."""
        super().__init__(input_tensor)

    def apply(self, grad_output):
        """
        Compute gradient for GELU.
        """
        
        tensor, = self.saved_tensors

        if isinstance(tensor, Tensor) and tensor.requires_grad:
            x = tensor.data
            # GELU derivative using the sigmoid approximation (matches forward):
            # forward: gelu(x) = x * sigmoid(1.702 * x)
            # d/dx [x * sig(1.702x)] = sig(1.702x) + x * 1.702 * sig(1.702x) * (1 - sig(1.702x))
            sig = 1.0 / (1.0 + np.exp(-1.702 * x))
            gelu_grad = sig + x * 1.702 * sig * (1.0 - sig)

            return (grad_output * gelu_grad,)
        return (None,)
        

In [26]:
#| export
class MSEBackward(Function):
    """
    Gradient computation for Mean Squared Error Loss.

    MSE: L = mean((predictions - targets)²)
    Derivative: ∂L/∂predictions = 2 * (predictions - targets) / N
    """

    def __init__(self, predictions, targets):
        """Initialize with predictions and targets."""
        super().__init__(predictions)
        self.targets_data = targets.data
        self.num_samples = np.size(targets.data)

    def apply(self, grad_output):
        """
        Compute gradient for MSE loss.
        """
        
        predictions, = self.saved_tensors

        if isinstance(predictions, Tensor) and predictions.requires_grad:
            # Gradient: 2 * (predictions - targets) / N
            grad = 2.0 * (predictions.data - self.targets_data) / self.num_samples

            return grad * grad_output,
        return None,
        

In [27]:
#| export
class BCEBackward(Function):
    """
    Gradient computation for Binary Cross-Entropy Loss.
    """

    def __init__(self, predictions, targets):
        """Initialize with predictions and targets."""
        super().__init__(predictions)
        self.targets_data = targets.data
        self.num_samples = np.size(targets.data)

    def apply(self, grad_output):
        """
        Compute gradient for BCE loss.
        """
        
        predictions, = self.saved_tensors

        if isinstance(predictions, Tensor) and predictions.requires_grad:
            eps = EPSILON
            p = np.clip(predictions.data, eps, 1 - eps)
            y = self.targets_data

            # Gradient: (p - y) / (p * (1-p) * N)
            grad = (p - y) / (p * (1 - p) * self.num_samples)

            return grad * grad_output,
        return None,
        

### Helper: Numerically Stable Softmax

Computing softmax naively as `exp(x) / sum(exp(x))` overflows for large values.
The fix is to subtract the maximum value first, which is mathematically equivalent
but numerically stable.

In [28]:
#| export
def _stable_softmax(logits_data):
    """
    Compute softmax probabilities with numerical stability.

    Subtracts the max value per row before exponentiating to prevent overflow.

    Args:
        logits_data: numpy array of shape (batch_size, num_classes)

    Returns:
        numpy array of softmax probabilities, same shape as input
    """
    
    max_logits = np.max(logits_data, axis=1, keepdims=True)
    exp_logits = np.exp(logits_data - max_logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
   

In [29]:
def test_unit_stable_softmax():
    """Test stable softmax helper."""
    print("Testing stable softmax helper...")

    # Basic correctness
    logits = np.array([[1.0, 2.0, 3.0]])
    probs = _stable_softmax(logits)
    assert np.allclose(probs.sum(axis=1), 1.0), f"Softmax should sum to 1, got {probs.sum(axis=1)}"

    # Verify correct values
    expected = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
    assert np.allclose(probs, expected), f"Softmax values wrong: {probs}"

    # Numerical stability: large values that would overflow naive exp()
    large_logits = np.array([[1000.0, 1001.0, 1002.0]])
    probs_large = _stable_softmax(large_logits)
    assert not np.any(np.isnan(probs_large)), "Stable softmax should handle large values"
    assert not np.any(np.isinf(probs_large)), "Stable softmax should not overflow"
    assert np.allclose(probs_large.sum(axis=1), 1.0), "Large softmax should sum to 1"

    # Batch dimension
    batch_logits = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    batch_probs = _stable_softmax(batch_logits)
    assert batch_probs.shape == (3, 2), f"Expected (3, 2), got {batch_probs.shape}"
    assert np.allclose(batch_probs.sum(axis=1), np.ones(3)), "Each row should sum to 1"

    print("  Stable softmax helper works correctly!")

if __name__ == "__main__":
    test_unit_stable_softmax()

Testing stable softmax helper...
  Stable softmax helper works correctly!


### Helper: One-Hot Encoding

Converts class indices to one-hot vectors. This is needed by the cross-entropy
gradient formula: `grad = softmax - one_hot`.

In [30]:
#| export
def _one_hot_encode(targets, batch_size, num_classes):
    """
    Convert class indices to one-hot vectors.

    Args:
        targets: numpy array of integer class indices, shape (batch_size,)
        batch_size: number of samples
        num_classes: number of classes

    Returns:
        numpy array of one-hot vectors, shape (batch_size, num_classes)
    """
    
    one_hot = np.zeros((batch_size, num_classes), dtype=np.float32)
    one_hot[np.arange(batch_size), targets] = 1.0
    return one_hot
    

### Unit Test: One-Hot Encoding Helper

In [31]:
def test_unit_one_hot_encode():
    """Test one-hot encoding helper."""
    print("Testing one-hot encoding helper...")

    # Basic test
    targets = np.array([0, 2, 1])
    result = _one_hot_encode(targets, batch_size=3, num_classes=3)
    expected = np.array([[1, 0, 0], [0, 0, 1], [0, 1, 0]], dtype=np.float32)
    assert np.allclose(result, expected), f"One-hot encoding wrong: {result}"

    # Single sample
    result_single = _one_hot_encode(np.array([1]), batch_size=1, num_classes=4)
    assert result_single.shape == (1, 4), f"Expected (1, 4), got {result_single.shape}"
    assert result_single[0, 1] == 1.0, "Target class should be 1.0"
    assert result_single.sum() == 1.0, "Should have exactly one 1.0"

    # Each row sums to 1
    targets_batch = np.array([0, 1, 2, 3, 4])
    result_batch = _one_hot_encode(targets_batch, batch_size=5, num_classes=5)
    assert np.allclose(result_batch.sum(axis=1), np.ones(5)), "Each row should sum to 1"

    print("  One-hot encoding helper works correctly!")

if __name__ == "__main__":
    test_unit_one_hot_encode()

Testing one-hot encoding helper...
  One-hot encoding helper works correctly!


### CrossEntropyBackward - Gradient Rules for Cross-Entropy Loss

The cross-entropy gradient combines three sub-computations:
1. **Stable softmax**: Convert raw logits to probabilities
2. **One-hot encoding**: Convert target indices to indicator vectors
3. **Gradient formula**: `(softmax - one_hot) / batch_size`

In [32]:
#| export
class CrossEntropyBackward(Function):
    """
    Gradient computation for Cross-Entropy Loss.
    """

    def __init__(self, logits, targets):
        """Initialize with logits and target class indices."""
        super().__init__(logits)
        self.targets_data = targets.data.astype(int)
        self.batch_size = logits.data.shape[0]
        self.num_classes = logits.data.shape[1]

    def apply(self, grad_output):
        """
        Compute gradient for cross-entropy loss.
        """
        
        logits, = self.saved_tensors

        if isinstance(logits, Tensor) and logits.requires_grad:
            softmax = _stable_softmax(logits.data)
            one_hot = _one_hot_encode(self.targets_data, self.batch_size, self.num_classes)

            # Gradient: (softmax - one_hot) / batch_size
            grad = (softmax - one_hot) / self.batch_size

            return grad * grad_output,
        return None,
        

In [33]:
#| export
# ===== Global Gradient Tracking Flag =====
# Why this exists: During inference or parameter updates, we don't need to build
# computation graphs. Skipping graph construction saves memory and time.
# This matches PyTorch's torch.no_grad() behavior.
_GRAD_TRACKING_ENABLED = True


def is_grad_enabled():
    """Check if gradient tracking is currently enabled.

    Returns True when operations should build computation graphs,
    False when inside a no_grad() context.
    """
    return _GRAD_TRACKING_ENABLED


class no_grad:
    """Context manager that disables gradient tracking.

    When entering this context, all operations will skip computation graph
    construction — tensors produced inside will have requires_grad=False
    regardless of their inputs. This is essential for:

    1. **Inference**: No need to track gradients when making predictions
    2. **Parameter updates**: Optimizers modify weights without recording history
    3. **Memory savings**: Skipping graph construction reduces memory usage
    """

    def __enter__(self):
        """Save previous state and disable gradient tracking."""
        global _GRAD_TRACKING_ENABLED
        # Save previous state so nested contexts restore correctly
        self._prev_state = _GRAD_TRACKING_ENABLED
        _GRAD_TRACKING_ENABLED = False
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Restore previous gradient tracking state."""
        global _GRAD_TRACKING_ENABLED
        _GRAD_TRACKING_ENABLED = self._prev_state
        return False  # Don't suppress exceptions

In [35]:
#| export
def enable_autograd(quiet=False):
    """
    Enable gradient tracking for all Tensor operations.

    This function enhances the existing Tensor class with autograd capabilities.
    Call this once to activate gradients globally.

    **Args:**
        quiet (bool): If True, suppress status messages. Default: False.

    **What it does:**
    - Replaces Tensor operations with gradient-tracking versions
    - Adds backward() method for reverse-mode differentiation
    - Enables computation graph building
    - Maintains full backward compatibility

    **After calling this:**
    - Tensor operations will track computation graphs
    - backward() method becomes available
    - Gradients will flow through operations
    - requires_grad=True enables tracking per tensor

    **Example:**
    ```python
    enable_autograd()  # Call once
    x = Tensor([2.0], requires_grad=True)
    y = x * 3
    y.backward()
    print(x.grad)  # [3.0]
    ```
    """

    
    if hasattr(Tensor, '_autograd_enabled'):
        # Silently return if already enabled - no need to warn
        return

    # ===== STEP 1: Add gradient infrastructure to Tensor =====
    # Store original __init__ to extend it
    _original_init = Tensor.__init__

    def gradient_aware_init(self, data, requires_grad=False):
        """Extended Tensor init that supports gradient tracking."""
        _original_init(self, data)
        self.requires_grad = requires_grad
        self.grad = None

    # Replace __init__ with gradient-aware version
    Tensor.__init__ = gradient_aware_init

    # Store original operations
    # These are guaranteed to exist from Module 01 (Tensor class)
    _original_add = Tensor.__add__
    _original_sub = Tensor.__sub__
    _original_mul = Tensor.__mul__
    _original_div = Tensor.__truediv__
    _original_getitem = Tensor.__getitem__

    # These methods are also guaranteed from Module 01 - trust Single Tensor Class
    _original_matmul = Tensor.matmul
    _original_transpose = Tensor.transpose
    _original_reshape = Tensor.reshape

    # Helper to safely check requires_grad (handles tensors created before enable_autograd)
    def _get_requires_grad(tensor):
        """Safely get requires_grad, defaulting to False for pre-autograd tensors.

        Also returns False when inside a no_grad() context, since we should
        not build computation graphs there.
        """
        if not _GRAD_TRACKING_ENABLED:
            return False
        return getattr(tensor, 'requires_grad', False) if isinstance(tensor, Tensor) else False

    def _ensure_grad_attrs(tensor):
        """Ensure tensor has gradient attributes (for tensors created before enable_autograd)."""
        if isinstance(tensor, Tensor):
            if not hasattr(tensor, 'requires_grad'):
                tensor.requires_grad = False
            if not hasattr(tensor, 'grad'):
                tensor.grad = None

    # Enhanced operations that track gradients
    def tracked_add(self, other):
        """
        Addition with gradient tracking.

        Enhances the original __add__ method to build computation graphs
        when requires_grad=True for any input.
        """
        # Ensure self has gradient attributes
        _ensure_grad_attrs(self)

        # Convert scalar to Tensor if needed
        if not isinstance(other, Tensor):
            other = Tensor(other)
        _ensure_grad_attrs(other)

        # Call original operation
        result = _original_add(self, other)
        _ensure_grad_attrs(result)

        # Track gradient if needed
        if _get_requires_grad(self) or _get_requires_grad(other):
            result.requires_grad = True
            result._grad_fn = AddBackward(self, other)

        return result

    def tracked_radd(self, other):
        """Scalar-left addition with gradient tracking."""
        return tracked_add(self, other)

    def tracked_mul(self, other):
        """
        Multiplication with gradient tracking.

        Enhances the original __mul__ method to build computation graphs
        when requires_grad=True for any input.
        """
        _ensure_grad_attrs(self)

        if not isinstance(other, Tensor):
            other_tensor = Tensor(other)
        else:
            other_tensor = other
        _ensure_grad_attrs(other_tensor)

        result = _original_mul(self, other)
        _ensure_grad_attrs(result)

        # Pass other_tensor (always a Tensor) so MulBackward can call .data on it.
        if _get_requires_grad(self) or _get_requires_grad(other_tensor):
            result.requires_grad = True
            result._grad_fn = MulBackward(self, other_tensor)

        return result

    def tracked_rmul(self, other):
        """Scalar-left multiplication with gradient tracking."""
        return tracked_mul(self, other)

    def tracked_matmul(self, other):
        """
        Matrix multiplication with gradient tracking.

        Enhances the original matmul method to build computation graphs
        when requires_grad=True for any input.
        """
        _ensure_grad_attrs(self)
        _ensure_grad_attrs(other)

        # Call original matmul from Module 01
        result = _original_matmul(self, other)
        _ensure_grad_attrs(result)

        # Track gradient if needed
        if _get_requires_grad(self) or _get_requires_grad(other):
            result.requires_grad = True
            result._grad_fn = MatmulBackward(self, other)

        return result

    def tracked_transpose(self, dim0=None, dim1=None):
        """
        Transpose with gradient tracking.

        Enhances the original transpose method to build computation graphs
        when requires_grad=True for the input.
        """
        _ensure_grad_attrs(self)

        # Call original transpose from Module 01
        result = _original_transpose(self, dim0, dim1)
        _ensure_grad_attrs(result)

        # Track gradient if needed
        if _get_requires_grad(self):
            result.requires_grad = True
            result._grad_fn = TransposeBackward(self, dim0, dim1)

        return result

    def tracked_reshape(self, *shape):
        """
        Reshape with gradient tracking.

        Enhances the original reshape method to build computation graphs
        when requires_grad=True for the input.
        """
        _ensure_grad_attrs(self)
        original_shape = self.shape

        # Call original reshape from Module 01
        result = _original_reshape(self, *shape)
        _ensure_grad_attrs(result)

        # Track gradient if needed
        if _get_requires_grad(self):
            result.requires_grad = True
            result._grad_fn = ReshapeBackward(self, original_shape)

        return result

    def tracked_sub(self, other):
        """
        Subtraction with gradient tracking.

        Enhances the original __sub__ method to build computation graphs
        when requires_grad=True for any input.
        """
        _ensure_grad_attrs(self)

        # Convert scalar to Tensor if needed
        if not isinstance(other, Tensor):
            other = Tensor(other)
        _ensure_grad_attrs(other)

        # Call original operation
        result = _original_sub(self, other)
        _ensure_grad_attrs(result)

        # Track gradient if needed
        if _get_requires_grad(self) or _get_requires_grad(other):
            result.requires_grad = True
            result._grad_fn = SubBackward(self, other)

        return result

    def tracked_rsub(self, other):
        """Scalar-left subtraction with gradient tracking."""
        if not isinstance(other, Tensor):
            other = Tensor(other)
        return tracked_sub(other, self)

    def tracked_div(self, other):
        """
        Division with gradient tracking.

        Enhances the original __truediv__ method to build computation graphs
        when requires_grad=True for any input.
        """
        _ensure_grad_attrs(self)

        # Convert scalar to Tensor if needed
        if not isinstance(other, Tensor):
            other = Tensor(other)
        _ensure_grad_attrs(other)

        # Call original operation
        result = _original_div(self, other)
        _ensure_grad_attrs(result)

        # Track gradient if needed
        if _get_requires_grad(self) or _get_requires_grad(other):
            result.requires_grad = True
            result._grad_fn = DivBackward(self, other)

        return result

    def tracked_rdiv(self, other):
        """Scalar-left division with gradient tracking."""
        if not isinstance(other, Tensor):
            other = Tensor(other)
        return tracked_div(other, self)

    def tracked_getitem(self, key):
        """
        Indexing/slicing with gradient tracking.

        Enhances the original __getitem__ method to build computation graphs
        when requires_grad=True for the input.
        """
        _ensure_grad_attrs(self)

        # Call original __getitem__ from Module 01
        result = _original_getitem(self, key)
        _ensure_grad_attrs(result)

        # Track gradient if needed
        if _get_requires_grad(self):
            result.requires_grad = True
            result._grad_fn = SliceBackward(self, key)

        return result

    def sum_op(self, axis=None, keepdims=False):
        """
        Sum operation with gradient tracking.

        Creates a new sum method that builds computation graphs
        when requires_grad=True.
        """
        _ensure_grad_attrs(self)

        result_data = np.sum(self.data, axis=axis, keepdims=keepdims)
        result = Tensor(result_data)

        if _get_requires_grad(self):
            result.requires_grad = True
            result._grad_fn = SumBackward(self, axis=axis, keepdims=keepdims)

        return result

    def backward(self, gradient=None, retain_graph=False):
        """
        Compute gradients via backpropagation.

        This is the key method that makes training possible!
        It implements reverse-mode automatic differentiation.
        """
        # Ensure gradient attributes exist
        _ensure_grad_attrs(self)

        # Only compute gradients if required
        if not _get_requires_grad(self):
            return

        # Initialize gradient if not provided (for scalar outputs)
        if gradient is None:
            if self.data.size == 1:
                gradient = np.ones_like(self.data)
            else:
                raise ValueError(
                    f"backward() called on non-scalar tensor without gradient argument.\n"
                    f"  Tensor shape: {self.shape}\n"
                    f"  Issue: For non-scalar outputs, you must provide the gradient from the next layer.\n"
                    f"  Fix: Call backward(gradient) with the gradient tensor from the loss function."
                )

        # Initialize or accumulate gradient
        if self.grad is None:
            self.grad = np.zeros_like(self.data)

        # Handle broadcasting: sum gradient to match self.data shape
        # This happens when operations broadcast tensors (e.g., adding bias to batch)
        if gradient.shape != self.grad.shape:
            # Step 1: Remove extra leading dimensions added during forward pass
            # Example: gradient (batch_size, features) → self.grad (features,)
            while gradient.ndim > self.grad.ndim:
                gradient = gradient.sum(axis=0)

            # Step 2: Sum over dimensions that were size-1 in original tensor
            # Example: bias with shape (1,) broadcast to (batch_size,) during forward
            for i in range(gradient.ndim):
                if self.grad.shape[i] == 1 and gradient.shape[i] != 1:
                    gradient = gradient.sum(axis=i, keepdims=True)

        self.grad += gradient

        # Propagate gradients through computation graph
        # _grad_fn is set by autograd enhancement when tensor is created from an operation
        grad_fn = getattr(self, '_grad_fn', None)
        if grad_fn is not None:
            grads = grad_fn.apply(gradient)

            # Recursively call backward on parent tensors
            for tensor, grad in zip(grad_fn.saved_tensors, grads):
                if isinstance(tensor, Tensor) and tensor.requires_grad and grad is not None:
                    tensor.backward(grad, retain_graph=retain_graph)

            # Release computation graph to free memory (matches PyTorch's default)
            # Why: The graph stores references to all intermediate tensors. Without
            # cleanup, these references prevent garbage collection, causing memory
            # to grow linearly with the number of training steps.
            if not retain_graph:
                self._grad_fn = None

    def zero_grad(self):
        """
        Reset gradients to zero.

        Call this before each backward pass to prevent gradient accumulation
        from previous iterations.
        """
        self.grad = None

    # Install enhanced operations
    Tensor.__add__ = tracked_add
    Tensor.__radd__ = tracked_radd
    Tensor.__sub__ = tracked_sub
    Tensor.__rsub__ = tracked_rsub
    Tensor.__mul__ = tracked_mul
    Tensor.__rmul__ = tracked_rmul
    Tensor.__truediv__ = tracked_div
    Tensor.__rtruediv__ = tracked_rdiv
    Tensor.__getitem__ = tracked_getitem
    Tensor.matmul = tracked_matmul
    Tensor.transpose = tracked_transpose
    Tensor.reshape = tracked_reshape
    Tensor.sum = sum_op
    Tensor.backward = backward
    Tensor.zero_grad = zero_grad

    # Patch activations and losses to track gradients
    try:
        from tinytorch.foundation.activations import Sigmoid, ReLU, Softmax, GELU, Tanh
        from tinytorch.foundation.losses import BinaryCrossEntropyLoss, MSELoss, CrossEntropyLoss

        # Store original methods
        _original_sigmoid_forward = Sigmoid.forward
        _original_relu_forward = ReLU.forward
        _original_softmax_forward = Softmax.forward
        _original_gelu_forward = GELU.forward
        _original_tanh_forward = Tanh.forward
        _original_bce_forward = BinaryCrossEntropyLoss.forward
        _original_mse_forward = MSELoss.forward
        _original_ce_forward = CrossEntropyLoss.forward

        def tracked_sigmoid_forward(self, x):
            """Sigmoid with gradient tracking."""
            result_data = 1.0 / (1.0 + np.exp(-x.data))
            result = Tensor(result_data)

            if _GRAD_TRACKING_ENABLED and x.requires_grad:
                result.requires_grad = True
                result._grad_fn = SigmoidBackward(x, result)

            return result

        def tracked_tanh_forward(self, x):
            """Tanh with gradient tracking."""
            result_data = np.tanh(x.data)
            result = Tensor(result_data)

            if _GRAD_TRACKING_ENABLED and x.requires_grad:
                result.requires_grad = True
                result._grad_fn = TanhBackward(x, result)

            return result

        def tracked_relu_forward(self, x):
            """ReLU with gradient tracking."""
            result_data = np.maximum(0, x.data)
            result = Tensor(result_data)

            if _GRAD_TRACKING_ENABLED and x.requires_grad:
                result.requires_grad = True
                result._grad_fn = ReLUBackward(x)

            return result

        def tracked_softmax_forward(self, x, dim=-1):
            """Softmax with gradient tracking."""
            # Call original forward to get result using Tensor operations
            result = _original_softmax_forward(self, x, dim=dim)

            # Attach the correct gradient function
            if _GRAD_TRACKING_ENABLED and x.requires_grad:
                result.requires_grad = True
                result._grad_fn = SoftmaxBackward(x, result, dim)

            return result

        def tracked_gelu_forward(self, x):
            """GELU with gradient tracking."""
            # Call original forward to get result
            result = _original_gelu_forward(self, x)

            # Attach the correct gradient function
            if _GRAD_TRACKING_ENABLED and x.requires_grad:
                result.requires_grad = True
                result._grad_fn = GELUBackward(x)

            return result

        def tracked_bce_forward(self, predictions, targets):
            """Binary cross-entropy with gradient tracking."""
            # Compute BCE loss
            eps = EPSILON
            clamped_preds = np.clip(predictions.data, eps, 1 - eps)
            log_preds = np.log(clamped_preds)
            log_one_minus_preds = np.log(1 - clamped_preds)
            bce_per_sample = -(targets.data * log_preds + (1 - targets.data) * log_one_minus_preds)
            bce_loss = np.mean(bce_per_sample)

            result = Tensor(bce_loss)

            if _GRAD_TRACKING_ENABLED and predictions.requires_grad:
                result.requires_grad = True
                result._grad_fn = BCEBackward(predictions, targets)

            return result

        def tracked_mse_forward(self, predictions, targets):
            """MSE loss with gradient tracking."""
            # Compute MSE loss
            diff = predictions.data - targets.data
            squared_diff = diff ** 2
            mse = np.mean(squared_diff)

            result = Tensor(mse)

            if _GRAD_TRACKING_ENABLED and predictions.requires_grad:
                result.requires_grad = True
                result._grad_fn = MSEBackward(predictions, targets)

            return result

        def tracked_ce_forward(self, logits, targets):
            """Cross-entropy loss with gradient tracking."""
            from tinytorch.core.losses import log_softmax

            # Compute log-softmax for numerical stability
            log_probs = log_softmax(logits, dim=-1)

            # Select log-probabilities for correct classes
            batch_size = logits.shape[0]
            target_indices = targets.data.astype(int)
            selected_log_probs = log_probs.data[np.arange(batch_size), target_indices]

            # Return negative mean
            ce_loss = -np.mean(selected_log_probs)

            result = Tensor(ce_loss)

            if _GRAD_TRACKING_ENABLED and logits.requires_grad:
                result.requires_grad = True
                result._grad_fn = CrossEntropyBackward(logits, targets)

            return result

        # Install patched methods
        Sigmoid.forward = tracked_sigmoid_forward
        ReLU.forward = tracked_relu_forward
        Softmax.forward = tracked_softmax_forward
        GELU.forward = tracked_gelu_forward
        Tanh.forward = tracked_tanh_forward
        BinaryCrossEntropyLoss.forward = tracked_bce_forward
        MSELoss.forward = tracked_mse_forward
        CrossEntropyLoss.forward = tracked_ce_forward

    except ImportError:
        # Activations/losses not yet available (happens during module development)
        pass

    # Mark as enabled
    Tensor._autograd_enabled = True

    if not quiet:
        print("✅ Autograd enabled! Tensors now track gradients.")
        print("   - Operations build computation graphs")
        print("   - backward() computes gradients")
        print("   - requires_grad=True enables tracking")

# Auto-enable when module is imported
# Always quiet to avoid cluttering user imports
import os
enable_autograd(quiet=True)

###  Unit Test: Tensor Autograd Enhancement

This test validates our enhanced Tensor class computes gradients correctly.


In [36]:
def test_unit_tensor_autograd():
    """ Test Tensor autograd enhancement."""
    print(" Unit Test: Tensor Autograd Enhancement...")

    # Test simple gradient computation
    x = Tensor([2.0], requires_grad=True)
    y = x * 3
    z = y + 1  # z = 3x + 1, so dz/dx = 3

    z.backward()
    assert np.allclose(x.grad, [3.0]), f"Expected [3.0], got {x.grad}"

    # Test matrix multiplication gradients
    a = Tensor([[1.0, 2.0]], requires_grad=True)  # 1x2
    b = Tensor([[3.0], [4.0]], requires_grad=True)  # 2x1
    c = a.matmul(b)  # 1x1, result = [[11.0]]

    c.backward()
    assert np.allclose(a.grad, [[3.0, 4.0]]), f"Expected [[3.0, 4.0]], got {a.grad}"
    assert np.allclose(b.grad, [[1.0], [2.0]]), f"Expected [[1.0], [2.0]], got {b.grad}"

    # Test computation graph with multiple operations
    x = Tensor([1.0, 2.0], requires_grad=True)
    y = x * 2      # y = [2, 4]
    z = y.sum()    # z = 6

    z.backward()
    assert np.allclose(x.grad, [2.0, 2.0]), f"Expected [2.0, 2.0], got {x.grad}"

    print("✅ Tensor autograd enhancement works correctly!")

if __name__ == "__main__":
    test_unit_tensor_autograd()

 Unit Test: Tensor Autograd Enhancement...
✅ Tensor autograd enhancement works correctly!


##  Systems Analysis: Computation Graph Memory

Let's understand ONE key systems concept: **computation graph memory overhead**.


In [40]:
def analyze_computation_graph_memory():
    """ Demonstrate memory overhead of computation graphs."""
    print(" Analyzing Computation Graph Memory...")
    print("=" * 60)

    import sys

    # Create tensors with different sizes
    sizes = [(100, 100), (500, 500), (1000, 1000)]

    print("\nMemory comparison: With vs Without Gradient Tracking")
    print("-" * 60)

    for shape in sizes:
        # Without gradient tracking
        x_no_grad = Tensor(rng.standard_normal(shape))
        base_memory = x_no_grad.data.nbytes

        # With gradient tracking
        x_with_grad = Tensor(rng.standard_normal(shape), requires_grad=True)
        y = x_with_grad * 2  # Simple operation that builds graph
        z = y + 1

        # Estimate graph overhead: saved tensors in grad_fn
        graph_overhead = 0
        if hasattr(z, '_grad_fn') and z._grad_fn is not None:
            for tensor in z._grad_fn.saved_tensors:
                if isinstance(tensor, Tensor):
                    graph_overhead += tensor.data.nbytes

        print(f"\nShape {shape}:")
        print(f"   Base tensor: {base_memory / 1024:.1f} KB")
        print(f"   Graph overhead: {graph_overhead / 1024:.1f} KB")
        print(f"   Overhead ratio: {(graph_overhead / base_memory):.1f}x")

    print("\n" + "=" * 60)
    print("📊 KEY INSIGHTS:")
    print("   1. Each operation saves inputs for backward pass")
    print("   2. Memory scales with number of operations, not just parameters")
    print("   3. Deep networks have more graph overhead than shallow ones")
    print("   4. This is why requires_grad is opt-in, not default!")

    print("\n🚀 REAL-WORLD IMPLICATIONS:")
    print("   - Training uses ~2-3x memory of inference")
    print("   - torch.no_grad() context saves memory during evaluation")

    print("\n" + "=" * 60)

# Run the systems analysis
if __name__ == "__main__":
    analyze_computation_graph_memory()

 Analyzing Computation Graph Memory...

Memory comparison: With vs Without Gradient Tracking
------------------------------------------------------------

Shape (100, 100):
   Base tensor: 39.1 KB
   Graph overhead: 39.1 KB
   Overhead ratio: 1.0x

Shape (500, 500):
   Base tensor: 976.6 KB
   Graph overhead: 976.6 KB
   Overhead ratio: 1.0x

Shape (1000, 1000):
   Base tensor: 3906.2 KB
   Graph overhead: 3906.3 KB
   Overhead ratio: 1.0x

📊 KEY INSIGHTS:
   1. Each operation saves inputs for backward pass
   2. Memory scales with number of operations, not just parameters
   3. Deep networks have more graph overhead than shallow ones
   4. This is why requires_grad is opt-in, not default!

🚀 REAL-WORLD IMPLICATIONS:
   - Training uses ~2-3x memory of inference
   - torch.no_grad() context saves memory during evaluation



##  Module Integration Test

Final validation that everything works together correctly.

In [46]:
def test_module():
    """🧪 Module Test: Complete Integration

    Comprehensive test of entire module functionality.
    """
    print("🧪 RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests
    print("Running unit tests...")
    test_unit_stable_softmax()
    test_unit_one_hot_encode()
    test_unit_function_classes()
    test_unit_tensor_autograd()

    print("\nRunning integration scenarios...")

    # Test 1: Multi-layer computation graph
    print("🔬 Integration Test: Multi-layer Neural Network...")

    # Create a 3-layer computation: x -> Linear -> Linear -> Linear -> loss
    x = Tensor([[1.0, 2.0]], requires_grad=True)
    W1 = Tensor([[0.5, 0.3, 0.1], [0.2, 0.4, 0.6]], requires_grad=True)
    b1 = Tensor([[0.1, 0.2, 0.3]], requires_grad=True)

    # First layer
    h1 = x.matmul(W1) + b1
    assert h1.shape == (1, 3)
    assert h1.requires_grad == True

    # Second layer
    W2 = Tensor([[0.1], [0.2], [0.3]], requires_grad=True)
    h2 = h1.matmul(W2)
    assert h2.shape == (1, 1)

    # Compute simple loss (just square the output for testing)
    loss = h2 * h2

    # Backward pass
    loss.backward()

    # Verify all parameters have gradients
    assert x.grad is not None
    assert W1.grad is not None
    assert b1.grad is not None
    assert W2.grad is not None
    assert x.grad.shape == x.shape
    assert W1.grad.shape == W1.shape

    print("✅ Multi-layer neural network gradients work!")

    # Test 2: Gradient accumulation
    print("🔬 Integration Test: Gradient Accumulation...")

    x = Tensor([2.0], requires_grad=True)

    # First computation
    y1 = x * 3
    y1.backward()
    first_grad = x.grad.copy()

    # Second computation (should accumulate)
    y2 = x * 5
    y2.backward()

    assert np.allclose(x.grad, first_grad + 5.0), "Gradients should accumulate"
    print("✅ Gradient accumulation works!")

    # Test 3: Complex mathematical operations
    print("🔬 Integration Test: Complex Operations...")

    a = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
    b = Tensor([[2.0, 1.0], [1.0, 2.0]], requires_grad=True)

    # Complex computation: ((a @ b) + a) * b
    temp1 = a.matmul(b)  # Matrix multiplication
    temp2 = temp1 + a    # Addition
    result = temp2 * b   # Element-wise multiplication
    final = result.sum() # Sum reduction

    final.backward()

    assert a.grad is not None
    assert b.grad is not None
    assert a.grad.shape == a.shape
    assert b.grad.shape == b.shape

    print("✅ Complex mathematical operations work!")

    print("\n" + "=" * 50)
    print("🎉 ALL TESTS PASSED! Module ready for export.")

# Test function defined above, will be called in main block

In [47]:
# Run comprehensive module test
if __name__ == "__main__":
    test_module()

🧪 RUNNING MODULE INTEGRATION TEST
Running unit tests...
Testing stable softmax helper...
  Stable softmax helper works correctly!
Testing one-hot encoding helper...
  One-hot encoding helper works correctly!
 Unit Test: Function Classes...
✅ Function classes work correctly!
 Unit Test: Tensor Autograd Enhancement...
✅ Tensor autograd enhancement works correctly!

Running integration scenarios...
🔬 Integration Test: Multi-layer Neural Network...
✅ Multi-layer neural network gradients work!
🔬 Integration Test: Gradient Accumulation...
✅ Gradient accumulation works!
🔬 Integration Test: Complex Operations...
✅ Complex mathematical operations work!

🎉 ALL TESTS PASSED! Module ready for export.


##  Extra Moment: Gradients Flow Automatically

**What you built:** An autograd engine that computes gradients through computation graphs.

In [48]:
def demo_autograd():
    """ See gradients computed automatically."""
    print(" MOMENT: Gradients Flow Automatically")
    print("=" * 45)

    # Simple example: y = x^2, so dy/dx = 2x
    x = Tensor(np.array([3.0]), requires_grad=True)
    y = x * x  # y = x^2

    print(f"x = {x.data[0]}")
    print(f"y = x^2 = {y.data[0]}")

    # Backward pass computes gradient
    y.backward()

    # Show computed vs expected gradient
    expected_grad = 2 * x.data[0]
    print(f"\nExpected: dy/dx = 2x = 2 * {x.data[0]} = {expected_grad}")
    print(f"Computed: {x.grad[0]}")
    print(f"Match: {np.allclose(x.grad[0], expected_grad)}")

    print("\n✨ Gradients computed automatically—no manual derivatives!")

In [49]:
if __name__ == "__main__":
    test_module()
    print("\n")
    demo_autograd()

🧪 RUNNING MODULE INTEGRATION TEST
Running unit tests...
Testing stable softmax helper...
  Stable softmax helper works correctly!
Testing one-hot encoding helper...
  One-hot encoding helper works correctly!
 Unit Test: Function Classes...
✅ Function classes work correctly!
 Unit Test: Tensor Autograd Enhancement...
✅ Tensor autograd enhancement works correctly!

Running integration scenarios...
🔬 Integration Test: Multi-layer Neural Network...
✅ Multi-layer neural network gradients work!
🔬 Integration Test: Gradient Accumulation...
✅ Gradient accumulation works!
🔬 Integration Test: Complex Operations...
✅ Complex mathematical operations work!

🎉 ALL TESTS PASSED! Module ready for export.


 MOMENT: Gradients Flow Automatically
x = 3.0
y = x^2 = 9.0

Expected: dy/dx = 2x = 2 * 3.0 = 6.0
Computed: 6.0
Match: True

✨ Gradients computed automatically—no manual derivatives!


In [54]:
from nbdev.export import nb_export
nb_export('autograd.ipynb', lib_path='../tinytorch/')